# 01 · Data Foundations and Contracts

Build one deterministic multimodal retail world and prove its identities, foreign keys, missingness, imbalance, shapes, checksums, and privacy boundary.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Distinguish a data generator from a random table dump.
- Read schema, primary-key, foreign-key, label, and modality contracts.
- Measure missingness and imbalance before modeling.
- Explain why fixed seeds and file checksums are related but different reproducibility controls.
- Identify leakage risks before writing a model.

## World design

The portfolio profile contains 20,000 customers, 100,000 purchases, 50,000 browse events,
10,000 temporal sequences, 5,000 support messages, 5,000 images/bounding boxes, and a
customer–product bipartite graph. CI uses the same mechanisms at smaller scale.

The labels are deliberately learnable but noisy. That lets us detect broken training while still
retaining non-perfect metrics and failure cases.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

In [ ]:
import pandas as pd
import yaml

schema = yaml.safe_load(Path("data/schemas/retail_world.yaml").read_text())
display(pd.DataFrame(schema["tables"]["customers"]["columns"].items(), columns=["column", "contract"]))
schema["modalities"]

## Relational tables

Primary keys answer “which record is this?” Foreign keys answer “which entity does it belong to?”
Validation must test both uniqueness and referential integrity; checking only `not null` is not enough.

In [ ]:
customers = pd.read_csv(DATA_DIR / "customers.csv.gz")
transactions = pd.read_csv(DATA_DIR / "transactions.csv.gz")
views = pd.read_csv(DATA_DIR / "product_views.csv.gz")
texts = pd.read_csv(DATA_DIR / "support_texts.csv.gz")
display(customers.head())
display(views.head())
print({
    "unique_customer_pk": customers.customer_id.is_unique,
    "unique_transaction_pk": transactions.transaction_id.is_unique,
    "view_fk_valid": views.customer_id.isin(customers.customer_id).all(),
})

## Missing data

`income` and `loyalty_score` have injected missingness. Median imputation is robust and simple,
but it shrinks variance and can hide informative missingness. In production, ask whether data are
MCAR, MAR, or MNAR and consider a missing-indicator feature.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

missing = customers.isna().mean().sort_values(ascending=False)
display(missing[missing > 0].rename("missing_rate").to_frame())
ax = missing[missing > 0].plot.bar(title="Injected missingness")
ax.set_ylabel("fraction missing")
plt.show()

In [ ]:
from sklearn.impute import SimpleImputer

before = customers[["income", "loyalty_score"]].describe().T
imputed = SimpleImputer(strategy="median").fit_transform(customers[["income", "loyalty_score"]])
after = pd.DataFrame(imputed, columns=["income", "loyalty_score"]).describe().T
display(before.add_prefix("before_" ).join(after.add_prefix("after_")))

## Imbalanced targets

Churn and anomalous transactions are rare by design. A classifier that predicts zero for every
record can have high accuracy. Always report positive rate, PR-AUC, recall, and a confusion matrix.

In [ ]:
rates = pd.Series({
    "churn": customers.churned.mean(),
    "transaction_anomaly": transactions.is_anomaly.mean(),
    "view_conversion": views.converted.mean(),
    "positive_sentiment": texts.sentiment.mean(),
})
display(rates.rename("positive_rate").to_frame())
rates.plot.bar(title="Label prevalence across tasks")
plt.axhline(0.5, linestyle="--", color="grey")
plt.show()

## Browse-event mechanism

Conversion probability depends on dwell time and page depth through a logistic mechanism. This is
a known ground truth for debugging feature engineering; it is not a claim about real shoppers.

In [ ]:
summary = views.groupby(pd.qcut(views.dwell_seconds, 5, duplicates="drop"), observed=True).agg(
    conversion_rate=("converted", "mean"), mean_depth=("page_depth", "mean"), rows=("view_id", "size")
)
display(summary)
sns.lineplot(data=views.sample(min(1000, len(views)), random_state=20260820), x="dwell_seconds", y="converted")
plt.xscale("log")
plt.show()

## NumPy modalities

Tabular CSV files are not enough for RNN, vision, detection, and graph practice. NPZ preserves typed
arrays; JSON stores variable-length bounding-box records; the edge CSV remains auditable.

In [ ]:
import numpy as np

sequences = np.load(DATA_DIR / "sequences.npz")
images = np.load(DATA_DIR / "product_images.npz")
boxes = json.loads((DATA_DIR / "detection_labels.json").read_text())
edges = pd.read_csv(DATA_DIR / "graph_edges.csv.gz")
print({"sequence_x": sequences["x"].shape, "sequence_y": sequences["y"].shape})
print({"image_x": images["x"].shape, "image_y": images["y"].shape})
print({"boxes": len(boxes), "unique_edges": len(edges)})

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, image, label in zip(axes.flat, images["x"][:10], images["y"][:10], strict=True):
    axis.imshow(image.transpose(1, 2, 0))
    axis.set_title(f"class={label}")
    axis.axis("off")
plt.tight_layout()

## Determinism and integrity

A seed reproduces pseudo-random choices. A SHA-256 checksum proves that a concrete file has not
changed. The manifest versions the generator and records both controls.

In [ ]:
checksum_frame = pd.DataFrame(
    [{"file": name, "sha256_prefix": digest[:16]} for name, digest in manifest["sha256"].items()]
)
display(checksum_frame)
assert len(manifest["sha256"]) == 8
assert manifest["contains_real_pii"] is False

## Leakage checklist

- Split entities before fitting imputers/scalers.
- Put preprocessing inside cross-validation Pipelines.
- Do not aggregate future `day` values into past features.
- Do not let the same customer leak across entity-level evaluation when measuring new-customer performance.
- Keep label-generation columns out of model features unless the task explicitly permits them.

In [ ]:
print(Path("sql/sqlite/schema.sql").read_text())
print("MySQL and MSSQL versions use the same customer/view relationship and indexed access path.")

## Exercises and interview checkpoint

1. Double missingness and add a missing indicator; compare CV ROC-AUC.
2. Make conversion rarer and compare accuracy with PR-AUC.
3. Design an out-of-time split using `day`.
4. Explain why synthetic IDs are not automatically anonymous if generated from real source rows.
5. State exactly what the checksums prove—and what they do not prove.